In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

ACCESS_KEY = os.getenv("YC_ACCESS_KEY_ID")
SECRET_KEY = os.getenv("YC_SECRET_ACCESS_KEY")
REGION = os.getenv("YC_REGION")
ENDPOINT = os.getenv("YC_ENDPOINT")
BUCKET = os.getenv("YC_BUCKET")

In [3]:
import s3fs

fs = s3fs.S3FileSystem(
    key=ACCESS_KEY,
    secret=SECRET_KEY,
    client_kwargs={
        "endpoint_url": ENDPOINT,
        "region_name": REGION
    },
    config_kwargs={
        "signature_version": "s3v4",
        "s3": {"addressing_style": "path"}  # для Yandex важно
    }
)

In [4]:
print(fs.ls(BUCKET))

['binance-data-downloader/raw']


In [5]:
files = fs.glob(
    f"{BUCKET}/raw/klines/symbol=ADAUSDT/interval=1m/date=*/data.parquet"
)

print("Найдено файлов:", len(files))

Найдено файлов: 2193


In [6]:
import pandas as pd

dfs = []

for i, f in enumerate(files):
    with fs.open(f) as file:
        df_part = pd.read_parquet(file)
        dfs.append(df_part)

    if (i + 1) % 200 == 0:
        print(f"Прочитано {i+1} файлов")

df = pd.concat(dfs, ignore_index=True)

Прочитано 200 файлов
Прочитано 400 файлов
Прочитано 600 файлов
Прочитано 800 файлов
Прочитано 1000 файлов
Прочитано 1200 файлов
Прочитано 1400 файлов
Прочитано 1600 файлов
Прочитано 1800 файлов
Прочитано 2000 файлов


In [7]:
df['timestamp'] = pd.to_datetime(df['open_time'], unit='ms')
df = df.sort_values('timestamp')
df = df.set_index('timestamp')

In [8]:
dups = df.index.duplicated().sum()
print("Дубликатов:", dups)

Дубликатов: 0


In [23]:
df

,open_time,close_time,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote
timestamp,,,,,,,,,,,
2020-02-01 00:01:00,1580515260000,1580515319999,0.05382,0.05387,0.05381,0.05381,267470.0,14401.816406,25,67058.0,3609.846191
2020-02-01 00:02:00,1580515320000,1580515379999,0.05385,0.05386,0.05377,0.05377,85914.0,4625.166016,20,61800.0,3327.760010
2020-02-01 00:03:00,1580515380000,1580515439999,0.05377,0.05382,0.05368,0.05378,455970.0,24504.648438,49,232076.0,12473.500000
2020-02-01 00:04:00,1580515440000,1580515499999,0.05376,0.05382,0.05362,0.05368,90562.0,4861.035645,33,38391.0,2062.146484
2020-02-01 00:05:00,1580515500000,1580515559999,0.05364,0.05370,0.05360,0.05364,205360.0,11014.576172,30,66367.0,3560.824219
...,...,...,...,...,...,...,...,...,...,...,...
2026-02-01 23:55:00,1769990100000,1769990159999,0.28540,0.28610,0.28540,0.28550,1207482.0,345033.062500,675,451239.0,128884.609375
2026-02-01 23:56:00,1769990160000,1769990219999,0.28550,0.28570,0.28530,0.28570,323503.0,92375.164062,409,226803.0,64765.492188
2026-02-01 23:57:00,1769990220000,1769990279999,0.28560,0.28630,0.28560,0.28610,898333.0,256968.296875,637,533279.0,152532.250000


In [9]:
full_range = pd.date_range(df.index.min(), df.index.max(), freq='1min')
missing = full_range.difference(df.index)

print("Пропусков:", len(missing))

Пропусков: 898


In [24]:
df['date'] = df.index.date
counts = df.groupby('date').size()

print(counts.describe())

count    2193.000000
mean     1439.590059
std         0.491935
min      1439.000000
25%      1439.000000
50%      1440.000000
75%      1440.000000
max      1440.000000
dtype: float64


In [25]:
bad = df[
    (df['high'] < df['low']) |
    (df['close'] > df['high']) |
    (df['close'] < df['low'])
]

print("Плохих свечей:", len(bad))

Плохих свечей: 0


In [26]:
step_errors = (df.index.to_series().diff() != pd.Timedelta('1min')).sum()
print("Нарушений шага:", step_errors)

Нарушений шага: 899


In [27]:
existing_dates = set(
    [f.split("date=")[1].split("/")[0] for f in files]
)

expected_dates = set(
    pd.date_range(min(existing_dates), max(existing_dates)).strftime("%Y-%m-%d")
)

missing_days = expected_dates - existing_dates

print("Пропущенные дни:", len(missing_days))

Пропущенные дни: 0


In [28]:
df_trade = pd.read_csv(r'C:\projects\binance-dowloader-3.0\ADAUSDT-trades-2020-02-02.csv')

In [29]:
df_trade

,92341,0.05616,559.0,31.39344,1580601607778,true
0,92342,0.05615,218.0,12.24070,1580601611241,True
1,92343,0.05620,2897.0,162.81140,1580601616257,False
2,92344,0.05619,216.0,12.13704,1580601621313,False
3,92345,0.05620,4619.0,259.58780,1580601621313,False
4,92346,0.05620,4738.0,266.27560,1580601624720,False
...,...,...,...,...,...,...
49435,141796,0.05581,11610.0,647.95410,1580687982996,True
49436,141797,0.05583,1197.0,66.82851,1580687985693,False
49437,141798,0.05580,12120.0,676.29600,1580687987383,True
49438,141799,0.05580,17922.0,1000.04760,1580687987383,True


In [10]:
print(missing[:20])

DatetimeIndex(['2020-02-02', '2020-02-03', '2020-02-04', '2020-02-05',
               '2020-02-06', '2020-02-07', '2020-02-08', '2020-02-09',
               '2020-02-10', '2020-02-11', '2020-02-12', '2020-02-13',
               '2020-02-14', '2020-02-15', '2020-02-16', '2020-02-17',
               '2020-02-18', '2020-02-19', '2020-02-20', '2020-02-21'],
              dtype='datetime64[ns]', freq=None)


In [11]:
missing_df = pd.DataFrame({'ts': missing})
missing_df['date'] = missing_df['ts'].dt.date

print(missing_df['date'].value_counts().head(10))

date
2020-02-02    1
2020-02-03    1
2020-02-04    1
2020-02-05    1
2020-02-06    1
2020-02-07    1
2020-02-08    1
2020-02-09    1
2020-02-10    1
2020-02-11    1
Name: count, dtype: int64


In [12]:
counts = df.groupby(df.index.date).size()

bad_days = counts[counts != 1440]

print("Плохих дней:", len(bad_days))
print(bad_days.head())

Плохих дней: 899
2020-02-01    1439
2020-02-02    1439
2020-02-03    1439
2020-02-04    1439
2020-02-05    1439
dtype: int64


In [13]:
def validate_days(df):
    counts = df.groupby(df.index.date).size()

    missing_days = counts[counts < 1440]

    print("Дней с пропусками:", len(missing_days))
    return missing_days

In [16]:
def check_missing_in_day(day_df, date):
    start = pd.Timestamp(date)
    end = start + pd.Timedelta(days=1) - pd.Timedelta(minutes=1)

    full = pd.date_range(start, end, freq='1min')
    missing = full.difference(day_df.index)

    return missing

date = "2020-02-02"
sample_day = df[df.index.date == pd.to_datetime(date).date()]

missing = check_missing_in_day(sample_day, date)
print(missing)

DatetimeIndex(['2020-02-02 00:00:00'], dtype='datetime64[ns]', freq='min')


In [17]:
sample_day.index[0] - pd.to_datetime(sample_day.iloc[0]['open_time'], unit='ms')

Timedelta('0 days 00:00:00')

In [18]:
print(sample_day.iloc[0]['open_time'])

1580601660000.0


In [31]:
pd.to_datetime(1580601607778, unit='ms')

Timestamp('2020-02-02 00:00:07.778000')

In [20]:
sample_day.index[0]

Timestamp('2020-02-02 00:01:00')

In [21]:
pd.to_datetime(sample_day.iloc[0]['open_time'], unit='ms')

Timestamp('2020-02-02 00:01:00')